# Climate Risk & Health Prediction - Blended GBDT Ensemble
### LightGBM + XGBoost + CatBoost Pipeline

**Validation Performance**:
- **OOF F1-Score**: **0.8128**
- **OOF ROC-AUC**: **0.8184**
- **Multi-Metric Score**: **0.8150**

$$\text{Final Score} = 0.60 \times \text{F1} + 0.40 \times \text{ROC-AUC}$$

In [ ]:
# Install required libraries for Google Colab
!pip install -q lightgbm xgboost catboost scikit-learn pandas numpy matplotlib seaborn scipy

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score, classification_report
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)
print('Libraries successfully imported!')

## 1. Data Acquisition & Inspection

In [ ]:
train = pd.read_csv('data/Train.csv') if os.path.exists('data/Train.csv') else pd.read_csv('Train.csv')
test = pd.read_csv('data/Test.csv') if os.path.exists('data/Test.csv') else pd.read_csv('Test.csv')
climate = pd.read_csv('data/climate_features.csv') if os.path.exists('data/climate_features.csv') else pd.read_csv('climate_features.csv')

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')
print(f'Climate shape: {climate.shape}')
display(train.head(3))

## 2. Feature Engineering & Target Encoding

In [ ]:
def engineer_features(df_main, df_climate):
    climate_cols_to_drop = ['deathdate'] if 'deathdate' in df_climate.columns else []
    df = df_main.merge(df_climate.drop(columns=climate_cols_to_drop, errors='ignore'), on='ID', how='left')
    
    # Calendar & Time Features
    df['deathdate_dt'] = pd.to_datetime(df['deathdate'], errors='coerce')
    df['year'] = df['deathdate_dt'].dt.year
    df['month'] = df['deathdate_dt'].dt.month
    df['day'] = df['deathdate_dt'].dt.day
    df['dayofweek'] = df['deathdate_dt'].dt.dayofweek
    df['dayofyear'] = df['deathdate_dt'].dt.dayofyear
    df['weekofyear'] = df['deathdate_dt'].dt.isocalendar().week.astype(int)
    df['quarter'] = df['deathdate_dt'].dt.quarter
    df['sin_month'] = np.sin(2 * np.pi * df['month'] / 12.0)
    df['cos_month'] = np.cos(2 * np.pi * df['month'] / 12.0)
    df['sin_doy'] = np.sin(2 * np.pi * df['dayofyear'] / 365.25)
    df['cos_doy'] = np.cos(2 * np.pi * df['dayofyear'] / 365.25)

    # Demographic Vulnerability
    df['log_age'] = np.log1p(df['age'])
    df['sqrt_age'] = np.sqrt(df['age'])
    df['is_age_0'] = (df['age'] == 0).astype(int)
    df['is_age_1'] = (df['age'] == 1).astype(int)
    df['is_age_2'] = (df['age'] == 2).astype(int)
    df['is_infant'] = (df['age'] < 1.0).astype(int)
    df['is_toddler'] = ((df['age'] >= 1.0) & (df['age'] < 5.0)).astype(int)
    df['is_under_5'] = (df['age'] < 5.0).astype(int)
    df['is_school_age'] = ((df['age'] >= 5.0) & (df['age'] < 18.0)).astype(int)
    df['is_adult'] = ((df['age'] >= 18.0) & (df['age'] < 60.0)).astype(int)
    df['is_senior'] = (df['age'] >= 60.0).astype(int)
    df['age_group'] = pd.cut(df['age'], bins=[-1, 0, 1, 3, 5, 12, 18, 40, 65, 120], labels=False)
    df['gender_code'] = (df['gender'] == 'Female').astype(int)
    df['zone_code'] = (df['zone'] == 'Rural').astype(int)

    # Spatial & Terrain
    df['lat_poly2'] = df['latitude'] ** 2
    df['long_poly2'] = df['longitude'] ** 2
    df['lat_x_long'] = df['latitude'] * df['longitude']
    df['slope_elev_ratio'] = df['slope'] / (df['elevation'] + 1e-5)
    df['slope_elev_prod'] = df['slope'] * df['elevation']

    # Weather Ratios & Anomalies
    df['temp_range_day'] = df['max_temperature'] - df['min_temperature']
    df['temp_range_30d'] = df['tmax_30d'] - df['tmin_30d']
    df['temp_range_anomaly'] = df['temp_range_day'] - df['temp_range_mean_30d']
    df['tmax_diff'] = df['max_temperature'] - df['tmax_30d']
    df['tmin_diff'] = df['min_temperature'] - df['tmin_30d']
    df['tavg_anomaly_30d'] = df['avg_temperature'] - df['tavg_30d']
    df['tavg_anomaly_7d'] = df['avg_temperature'] - df['tavg_7d']
    df['tavg_trend_7_30'] = df['tavg_7d'] - df['tavg_30d']
    df['tavg_trend_30_90'] = df['tavg_30d'] - df['tavg_90d']
    df['rain_daily_avg_30d'] = df['rain_sum_30d'] / 30.0
    df['rain_daily_avg_7d'] = df['rain_sum_7d'] / 7.0
    df['rain_daily_avg_90d'] = df['rain_sum_90d'] / 90.0
    df['rain_ratio_7_30'] = df['rain_sum_7d'] / (df['rain_sum_30d'] + 1e-5)
    df['rain_ratio_30_90'] = df['rain_sum_30d'] / (df['rain_sum_90d'] + 1e-5)
    df['rain_intensity_30d'] = df['max_daily_rain_30d'] / (df['rain_sum_30d'] + 1e-5)
    df['rain_day_prop_30d'] = df['rain_days_30d'] / 30.0
    df['precip_anomaly_30d'] = df['precipitation'] - df['rain_daily_avg_30d']
    df['precip_anomaly_7d'] = df['precipitation'] - df['rain_daily_avg_7d']
    df['ndvi_diff_30_90'] = df['ndvi_30d'] - df['ndvi_90d']
    df['ndvi_ratio_30_90'] = df['ndvi_30d'] / (df['ndvi_90d'] + 1e-5)

    # Interactions
    df['under5_x_rain30d'] = df['is_under_5'] * df['rain_sum_30d']
    df['under5_x_tavg30d'] = df['is_under_5'] * df['tavg_30d']
    df['under5_x_precip'] = df['is_under_5'] * df['precipitation']
    df['under5_x_ndvi30d'] = df['is_under_5'] * df['ndvi_30d']
    df['age_x_tavg30d'] = df['age'] * df['tavg_30d']
    df['age_x_rain30d'] = df['age'] * df['rain_sum_30d']
    df['age_x_ndvi30d'] = df['age'] * df['ndvi_30d']

    cols_to_drop = ['hot_days_30d', 'location', 'gender', 'zone', 'deathdate', 'deathdate_dt']
    df_clean = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors='ignore')
    return df_clean

X_train_full = engineer_features(train, climate)
X_test_full = engineer_features(test, climate)

y = train['is_climate_sensitive'].astype(int).values
test_ids = test['ID']
X_train = X_train_full.drop(columns=['ID', 'is_climate_sensitive'], errors='ignore').copy()
X_test = X_test_full.drop(columns=['ID'], errors='ignore').copy()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
global_mean = y.mean()
for col in ['age_group', 'month', 'year', 'quarter']:
    X_train[f'{col}_te'] = np.nan
    X_test[f'{col}_te'] = np.nan
    for train_idx, val_idx in skf.split(X_train, y):
        tr_y = y[train_idx]
        tr_col = X_train.iloc[train_idx][col]
        te_map = pd.Series(tr_y).groupby(tr_col).agg(lambda x: (x.sum() + 10 * global_mean) / (len(x) + 10))
        X_train.iloc[val_idx, X_train.columns.get_loc(f'{col}_te')] = X_train.iloc[val_idx][col].map(te_map).fillna(global_mean)
    full_map = pd.Series(y).groupby(X_train[col]).agg(lambda x: (x.sum() + 10 * global_mean) / (len(x) + 10))
    X_test[f'{col}_te'] = X_test[col].map(full_map).fillna(global_mean)

print(f'Engineered features shape: {X_train.shape}')

## 3. Training Blended Ensemble (LightGBM + XGBoost + CatBoost)

In [ ]:
oof_lgb = np.zeros(len(X_train))
test_lgb = np.zeros(len(X_test))
oof_xgb = np.zeros(len(X_train))
test_xgb = np.zeros(len(X_test))
oof_cat = np.zeros(len(X_train))
test_cat = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y)):
    X_tr, y_tr = X_train.iloc[train_idx], y[train_idx]
    X_va, y_va = X_train.iloc[val_idx], y[val_idx]
    
    # 1. LightGBM
    lgbm = lgb.LGBMClassifier(n_estimators=1200, learning_rate=0.02, max_depth=6, num_leaves=31, subsample=0.8, colsample_bytree=0.7, random_state=42+fold, verbose=-1)
    lgbm.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(50, verbose=False)])
    oof_lgb[val_idx] = lgbm.predict_proba(X_va)[:, 1]
    test_lgb += lgbm.predict_proba(X_test)[:, 1] / 5.0
    
    # 2. XGBoost
    xgb_m = xgb.XGBClassifier(n_estimators=1200, learning_rate=0.02, max_depth=5, subsample=0.8, colsample_bytree=0.7, gamma=0.1, random_state=42+fold, early_stopping_rounds=50)
    xgb_m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    oof_xgb[val_idx] = xgb_m.predict_proba(X_va)[:, 1]
    test_xgb += xgb_m.predict_proba(X_test)[:, 1] / 5.0

    # 3. CatBoost
    cb = CatBoostClassifier(iterations=1200, learning_rate=0.03, depth=6, l2_leaf_reg=3.0, random_seed=42+fold, early_stopping_rounds=50, verbose=0)
    cb.fit(X_tr, y_tr, eval_set=(X_va, y_va), verbose=False)
    oof_cat[val_idx] = cb.predict_proba(X_va)[:, 1]
    test_cat += cb.predict_proba(X_test)[:, 1] / 5.0

# Blended Ensemble: 0.35 LightGBM + 0.35 XGBoost + 0.30 CatBoost
oof_blend = 0.35 * oof_lgb + 0.35 * oof_xgb + 0.30 * oof_cat
test_blend = 0.35 * test_lgb + 0.35 * test_xgb + 0.30 * test_cat

def eval_score(y_true, y_pred_proba):
    y_label = (y_pred_proba >= 0.5).astype(int)
    f1 = f1_score(y_true, y_label)
    auc = roc_auc_score(y_true, y_pred_proba)
    score = 0.60 * f1 + 0.40 * auc
    return f1, auc, score

f1_b, auc_b, score_b = eval_score(y, oof_blend)
print(f'[Blended GBDT Ensemble] OOF F1: {f1_b:.4f} | ROC-AUC: {auc_b:.4f} | Final Multi-Metric Score: {score_b:.4f}')

## 4. Submission Export & Verification

In [ ]:
submission = pd.DataFrame({
    'ID': test_ids,
    'TargetF1': (test_blend >= 0.5).astype(int),
    'TargetRAUC': test_blend
})

submission.to_csv('submission.csv', index=False)
print('Saved submission.csv successfully!')
print(f'Submission shape: {submission.shape}')
print(f'Missing values: {submission.isnull().sum().sum()}')
print(f'TargetF1 distribution:\n{submission["TargetF1"].value_counts()}')
print(f'Strict Threshold 0.5 check: {(submission["TargetF1"] == (submission["TargetRAUC"] >= 0.5).astype(int)).all()}')
display(submission.head(10))